# SATAY-ViT — Version 5 Training Notebook
**Architecture:** CLIP-seeded free `nn.Embedding` (identical structure to V2, better initialisation)

| Component | V2 | V5 |
|-----------|----|----|
| Task embedding | Random `nn.Embedding(15, 256)` | **CLIP-seeded** `nn.Embedding(15, 256)` |
| Embedding freedom | Fully trainable | Fully trainable |
| Projection layer | None | None |
| Eval floor | 0.0 | 0.0 (bug fixed) |

**Why CLIP seeding helps:** V2's random init produces mean inter-task cosine sim ≈ 0.  
CLIP seeding starts at ≈ 0.58 but backprop pushes it toward 0 faster, reaching a better minimum sooner.

## 1 · Imports & Configuration

In [6]:
import os, sys, json, torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# ── Paths ──────────────────────────────────────────────────────────
PIPELINE_DIR   = "e:/DVcon/DVcon/Pipelines/satay_vit"
VERSION_DIR    = os.path.join(PIPELINE_DIR, "Version_5")
DATA_ROOT      = "e:/DVcon/DVcon/Data_Preprocessed"
WEIGHTS_DIR    = os.path.join(VERSION_DIR, "weights_e2e")
KNOWLEDGE_PATH = os.path.join(WEIGHTS_DIR, "raw_knowledge_vectors.pt")

# ── Hyper-parameters ───────────────────────────────────────────────
EPOCHS        = 10
BATCH_SIZE    = 16
LR            = 2e-5
BACKBONE_LR   = 5e-7
FREEZE_EPOCHS = 5    # epochs with backbone frozen
GRAD_CLIP     = 0.5  # max gradient norm (tighter = less backbone drift)
EMBED_DIM     = 256

for p in [PIPELINE_DIR, VERSION_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.makedirs(WEIGHTS_DIR, exist_ok=True)

assert os.path.exists(KNOWLEDGE_PATH), (
    f"Raw CLIP knowledge vectors not found at:\n  {KNOWLEDGE_PATH}\n"
    "Run:  python generate_knowledge_embeddings.py"
)

# Sanity-check CLIP embedding separability (pre-training baseline)
import torch.nn.functional as F
raw = torch.load(KNOWLEDGE_PATH, weights_only=True)           # (14, 512)
raw_n = F.normalize(raw, dim=-1)
sim  = raw_n @ raw_n.T
mask = ~torch.eye(14, dtype=torch.bool)
print(f"Raw CLIP inter-task cosine sim: mean={sim[mask].mean():.4f}  (will improve after training)")
print(f"Knowledge vector shape: {tuple(raw.shape)}")
print()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Raw CLIP inter-task cosine sim: mean=0.5741  (will improve after training)
Knowledge vector shape: (14, 512)

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Device: cuda


## 2 · Dataset  (16 × 16 grid — same as V2/V4)

In [7]:
from utils.data_loader import COCOTasksDataset, custom_collate

train_ds = COCOTasksDataset(DATA_ROOT, split="train", grid_size=16)
val_ds   = COCOTasksDataset(DATA_ROOT, split="test",  grid_size=16)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, collate_fn=custom_collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, collate_fn=custom_collate)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

Loaded 100800 PREPROCESSED samples for train split. Target grid: 16x16
Loaded 12600 PREPROCESSED samples for test split. Target grid: 16x16
Train batches : 6300
Val   batches : 788


## 3 · Model

V5 is **architecturally identical to V2**.  
The only difference is that `task_embedding.weight` is seeded from a PCA-projected CLIP embedding  
instead of a random Gaussian, giving backprop a semantically structured starting point.

In [8]:
from Version_5.model_e2e import SATAYViT_E2E

model = SATAYViT_E2E(
    embed_dim      = EMBED_DIM,
    knowledge_path = KNOWLEDGE_PATH,
).to(device)

trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_pars = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:,}")
print(f"Total parameters     : {total_pars:,}")
print()

# Check initial embedding geometry
emb  = model.vit_head.task_embedding.weight[1:].detach()   # (14, 256)
emb_n = F.normalize(emb, dim=-1)
sim0  = emb_n @ emb_n.T
mask  = ~torch.eye(14, dtype=torch.bool)
print(f"Initial task_embedding cosine sim: mean={sim0[mask].mean():.4f}")
print("(Backprop will push this toward 0 — V2 converged to 0.02)")

  [MEViTHead] task_embedding seeded from CLIP (e:/DVcon/DVcon/Pipelines/satay_vit\Version_5\weights_e2e\raw_knowledge_vectors.pt)
Trainable parameters : 1,387,520
Total parameters     : 2,338,656

Initial task_embedding cosine sim: mean=0.5752
(Backprop will push this toward 0 — V2 converged to 0.02)


## 4 · Optimizer, Scheduler & Loss

In [9]:
criterion = nn.BCELoss()

# Two parameter groups (same as V2):
#   1. YOLO backbone  — lower lr  (pretrained, slow to change)
#   2. Everything else — full lr  (FPN, transformer, task_embedding)
backbone_params = list(model.backbone.parameters())
other_params    = [
    p for name, p in model.named_parameters()
    if p.requires_grad and "backbone" not in name
]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR, "weight_decay": 1e-4},
    {"params": other_params,    "lr": LR,          "weight_decay": 1e-4},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.05)

print("Optimizer  : AdamW, 2 param groups")
print(f"  backbone : lr={BACKBONE_LR}")
print(f"  head+emb : lr={LR}")
print(f"Scheduler  : CosineAnnealingLR, T_max={EPOCHS}")

Optimizer  : AdamW, 2 param groups
  backbone : lr=5e-07
  head+emb : lr=2e-05
Scheduler  : CosineAnnealingLR, T_max=10


## 5 · Training Loop

In [10]:
def set_backbone_grad(requires_grad: bool):
    for p in model.backbone.parameters():
        p.requires_grad = requires_grad

# ── Resume support ─────────────────────────────────────────────────
START_EPOCH   = 0
best_val_loss = float("inf")
train_history, val_history = [], []

ckpt_path  = os.path.join(WEIGHTS_DIR, "satay_vit_e2e_best.pt")
latest_path = os.path.join(WEIGHTS_DIR, "satay_vit_e2e_latest.pt")
hist_path   = os.path.join(WEIGHTS_DIR, "training_history.json")

if os.path.exists(latest_path):
    print(f"Resuming from {latest_path}")
    ck = torch.load(latest_path, map_location=device)
    START_EPOCH   = ck["epoch"]
    best_val_loss = ck.get("best_val_loss", float("inf"))
    model.load_state_dict(ck["state_dict"])
    optimizer.load_state_dict(ck["optimizer"])
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            h = json.load(f)
            train_history = h.get("train", [])
            val_history   = h.get("val",   [])
    print(f"  Resumed at epoch {START_EPOCH}, best_val={best_val_loss:.5f}")

# ── Phase setup ─────────────────────────────────────────────────────
if START_EPOCH < FREEZE_EPOCHS:
    set_backbone_grad(False)
    print(f"Phase 1: warming up ({FREEZE_EPOCHS - START_EPOCH} epochs, backbone frozen)")
else:
    set_backbone_grad(True)
    print("Phase 2: full end-to-end fine-tuning")

# ── Main loop ───────────────────────────────────────────────────────
for epoch in range(START_EPOCH, EPOCHS):

    if epoch == FREEZE_EPOCHS and START_EPOCH < FREEZE_EPOCHS:
        set_backbone_grad(True)
        print("\n>>> Phase 2: backbone unfrozen — end-to-end fine-tuning <<<")

    # Train
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for batch in pbar:
        imgs     = batch["image"].to(device)
        tasks    = batch["task_id"].to(device)
        gt_hmaps = batch["heatmap"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs, tasks), gt_hmaps)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_train = train_loss / len(train_loader)
    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ", leave=False):
            imgs     = batch["image"].to(device)
            tasks    = batch["task_id"].to(device)
            gt_hmaps = batch["heatmap"].to(device)
            val_loss += criterion(model(imgs, tasks), gt_hmaps).item()
    avg_val = val_loss / len(val_loader)

    train_history.append(avg_train)
    val_history.append(avg_val)
    phase = "frozen" if epoch < FREEZE_EPOCHS else "active"
    print(f"Epoch {epoch+1:>2}/{EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | backbone={phase}")

    # Save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            "epoch":          epoch + 1,
            "state_dict":     model.state_dict(),
            "optimizer":      optimizer.state_dict(),
            "best_val_loss":  best_val_loss,
            "knowledge_path": KNOWLEDGE_PATH,
        }, ckpt_path)
        print(f"  ==> Best checkpoint saved (val={best_val_loss:.5f})")

    # Save latest + per-epoch
    d = {
        "epoch":         epoch + 1,
        "state_dict":    model.state_dict(),
        "optimizer":     optimizer.state_dict(),
        "best_val_loss": best_val_loss,
    }
    torch.save(d, latest_path)
    torch.save(d, os.path.join(WEIGHTS_DIR, f"satay_vit_e2e_epoch_{epoch+1}.pt"))
    with open(hist_path, "w") as f:
        json.dump({"train": train_history, "val": val_history}, f)

print(f"\nTraining complete. Best val loss: {best_val_loss:.5f}")

Resuming from e:/DVcon/DVcon/Pipelines/satay_vit\Version_5\weights_e2e\satay_vit_e2e_latest.pt
  Resumed at epoch 3, best_val=0.10855
Phase 1: warming up (2 epochs, backbone frozen)


Epoch 4/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

Epoch 4/10 [Val]  :   0%|          | 0/788 [00:00<?, ?it/s]

Epoch  4/10 | Train: 0.1007 | Val: 0.1074 | backbone=frozen
  ==> Best checkpoint saved (val=0.10738)


Epoch 5/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

...

Epoch 6/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

Epoch 6/10 [Val]  :   0%|          | 0/788 [00:00<?, ?it/s]

Epoch  6/10 | Train: 0.0910 | Val: 0.1118 | backbone=active


Epoch 7/10 [Train]:   0%|          | 0/6300 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6 · Loss Curve

In [ ]:
from utils.plot_metrics import plot_training_losses
from IPython.display import Image, display

plot_path = plot_training_losses(train_history, val_history, save_dir=WEIGHTS_DIR)
display(Image(plot_path))

## 7 · Post-Training Task Vector Separability
The mean cosine similarity should be close to V2's **0.02** (nearly orthogonal).  
If it is still high (> 0.3), the model needs more epochs.

In [ ]:
import torch.nn.functional as F

model.eval()
with torch.no_grad():
    emb   = model.vit_head.task_embedding.weight[1:]   # (14, 256)
    emb_n = F.normalize(emb, dim=-1)
    sim   = emb_n @ emb_n.T

mask = ~torch.eye(14, dtype=torch.bool)
off  = sim[mask]
print("Post-training task_embedding cosine similarity:")
print(f"  Mean : {off.mean():.4f}  (V2 converged to 0.0205 — lower is better)")
print(f"  Min  : {off.min():.4f}")
print(f"  Max  : {off.max():.4f}")
if off.mean() < 0.15:
    print("  PASS — task vectors are well-separated!")
elif off.mean() < 0.35:
    print("  MARGINAL — consider 2-3 more epochs.")
else:
    print("  WARNING — task vectors still similar; check LR or run more epochs.")

## 8 · Evaluation — Top-1 Task-Aware Accuracy

In [ ]:
from utils.evaluate_map_variants import evaluate_best_model

evaluate_best_model(
    data_root         = DATA_ROOT,
    weights_path      = ckpt_path,
    output_dir        = VERSION_DIR,
    use_original_data = False,
    iou_thresholds    = [0.5, 0.75],
)